# Module 03 · Unit 03
# Bijections and Cryptographic Permutations

| | |
|---|---|
| **Estimated time** | 65–75 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Cryptographic Structure \[CS\] · Access Control \[AC\] |
| **Refresher** | Module 03 · Unit 02 — Relations and Functions |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Prove that a function is a bijection using the formal definition
2. Construct and apply the **inverse function** of a bijection
3. Compose bijections and prove the composition is also a bijection
4. Define a **permutation** and explain its role in symmetric cryptography
5. Apply the **Pigeonhole Principle** to reason about key spaces and hash collisions
6. Visualise a toy Substitution-Permutation Network (SPN) and connect it to the AES structure


## 🔣 Symbol Primer

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $f^{-1}$ | Inverse function | "$f$ inverse" | The function that undoes $f$: $f^{-1}(f(a)) = a$ |
| $g \circ f$ | Composition | "$g$ after $f$" | Apply $f$ first, then $g$: $(g \circ f)(x) = g(f(x))$ |
| $\text{id}_A$ | Identity function | "identity on $A$" | $\text{id}_A(a) = a$ for all $a \in A$ |
| $S_n$ | Symmetric group | "$S$ sub $n$" | The set of all permutations of $n$ elements |
| $\sigma, \pi, \tau$ | Permutation symbols | "sigma, pi, tau" | Conventional letters for permutation functions |
| $\lfloor x \rfloor$ | Floor | "floor of $x$" | Largest integer $\leq x$ |
| $\lceil x \rceil$ | Ceiling | "ceiling of $x$" | Smallest integer $\geq x$ |

> **The payoff of Module 03.** Every concept built in Units 01–02 converges here.
> A **permutation** is a bijection from a set to itself.
> A **block cipher** is a parameterised family of permutations.
> The **key space** is the set of all keys — its cardinality determines security.
> The **Pigeonhole Principle** explains why hash collisions must exist.
> Set theory, relations, and functions are not separate topics — they are three
> views of the same mathematical structure.

---


## 1 · Bijections — Definition and Proof Method

A function $f: A \to B$ is a **bijection** (also called a *one-to-one correspondence*)
if it is both injective and surjective:

$$f \text{ is bijective} \;\Longleftrightarrow\; f \text{ is injective} \wedge f \text{ is surjective}$$

Recall from Unit 02:
- **Injective:** $f(a_1) = f(a_2) \Rightarrow a_1 = a_2$ — distinct inputs, distinct outputs
- **Surjective:** $\forall b \in B,\; \exists a \in A,\; f(a) = b$ — every output is reached

### Proof Method for Bijections

The standard approach proves injectivity and surjectivity separately.

**Proving injective:** Assume $f(a_1) = f(a_2)$, then derive $a_1 = a_2$.

**Proving surjective:** Let $b \in B$ be arbitrary, construct an explicit $a \in A$
such that $f(a) = b$, and verify that $a \in A$.

---

### Example — The Byte Complement Is a Bijection

Define $f: \mathbb{B} \to \mathbb{B}$ on the byte set $\mathbb{B} = \{0, \ldots, 255\}$ by:

$$f(x) = 255 - x$$

**Proof that $f$ is a bijection.**

*Injective:* Assume $f(a_1) = f(a_2)$. Then $255 - a_1 = 255 - a_2$,
so $a_1 = a_2$. $\square$

*Surjective:* Let $b \in \mathbb{B}$ be arbitrary. Set $a = 255 - b$.
Then $a \in \{0, \ldots, 255\}$ (since $0 \leq 255 - b \leq 255$) and
$f(a) = 255 - (255 - b) = b$. $\square$

Therefore $f$ is a bijection. Its inverse is $f^{-1}(b) = 255 - b$ — the
same function. $f$ is its own inverse: $f \circ f = \text{id}_{\mathbb{B}}$.

**Cryptographic significance.** The NOT operation on bits is a bijection on
$\{0, 1\}$. The byte complement is a bijection on $\mathbb{B}$. These are the
simplest examples of the **substitution** layer in a block cipher.


## 2 · Inverse Functions and Composition

### The Inverse of a Bijection

If $f: A \to B$ is a bijection, its **inverse** $f^{-1}: B \to A$ is the unique
function satisfying:

$$f^{-1}(f(a)) = a \quad \forall a \in A \qquad\text{and}\qquad f(f^{-1}(b)) = b \quad \forall b \in B$$

Equivalently: $f^{-1} \circ f = \text{id}_A$ and $f \circ f^{-1} = \text{id}_B$.

**Key fact:** A function has an inverse if and only if it is a bijection.
Injectivity ensures the inverse is well-defined (no ambiguity).
Surjectivity ensures the inverse is total (defined for all $b \in B$).

**Cryptographic reading.** Encryption is $f$ (plaintext → ciphertext).
Decryption is $f^{-1}$ (ciphertext → plaintext). The requirement that decryption
always works and is unambiguous is exactly the requirement that encryption is
a bijection.

### Composition of Bijections

**Theorem.** If $f: A \to B$ and $g: B \to C$ are both bijections, then
$g \circ f: A \to C$ is also a bijection.

**Proof.**

*Injective:* Suppose $(g \circ f)(a_1) = (g \circ f)(a_2)$, i.e., $g(f(a_1)) = g(f(a_2))$.
Since $g$ is injective: $f(a_1) = f(a_2)$. Since $f$ is injective: $a_1 = a_2$. $\square$

*Surjective:* Let $c \in C$. Since $g$ is surjective, $\exists b \in B$ with $g(b) = c$.
Since $f$ is surjective, $\exists a \in A$ with $f(a) = b$. Then $(g \circ f)(a) = g(f(a)) = g(b) = c$. $\square$

**Cryptographic reading.** Composing two substitution layers both of which are
bijections gives a composed layer that is also a bijection — the ciphertext is
still uniquely decryptable. A **block cipher** is a long composition of bijective
layers (substitutions and permutations), and the full cipher is itself a bijection.


## 3 · Permutations

A **permutation** of a set $A$ is a bijection from $A$ to itself:

$$\sigma: A \to A \quad \text{bijective}$$

The set of all permutations of $A$ is denoted $S_{|A|}$ — the **symmetric group**.
If $|A| = n$, then $|S_n| = n!$ (n factorial) — the number of ways to arrange $n$
distinct elements.

### Why Permutations Are Central to Cryptography

A **substitution layer** in a block cipher is a permutation of the byte set
$\mathbb{B} = \{0, \ldots, 255\}$. The AES S-box is one specific permutation
of 256 elements — chosen for its resistance to differential and linear cryptanalysis.

A **permutation layer** in a block cipher rearranges the bit positions of the
block — it is a permutation of the index set $\{1, \ldots, n\}$ where $n$ is the
block size in bits.

Together, substitution + permutation layers give a cipher its **confusion**
(non-linear substitution) and **diffusion** (spreading bit influence across the
block). This is the **Substitution-Permutation Network (SPN)** structure
underlying AES, Present, and many other modern ciphers.

### Key Space as a Set

For a cipher with $k$-bit keys, the key space is $\mathcal{K} = \{0,1\}^k$
with $|\mathcal{K}| = 2^k$.

The cipher $E: \mathcal{K} \times \mathcal{M} \to \mathcal{C}$ takes a key and
plaintext to ciphertext. For each fixed key $\kappa \in \mathcal{K}$, the function
$E_\kappa: \mathcal{M} \to \mathcal{C}$ must be a bijection (so decryption works).

| Cipher | Key size $k$ | $|\mathcal{K}| = 2^k$ |
|---|---|---|
| DES (broken) | 56 bits | $\approx 7.2 \times 10^{16}$ |
| AES-128 | 128 bits | $\approx 3.4 \times 10^{38}$ |
| AES-256 | 256 bits | $\approx 1.2 \times 10^{77}$ |

Brute-force key search is an exhaustive traversal of the key space — the set
$\mathcal{K}$. The security of a cipher depends on $|\mathcal{K}|$ being
astronomically large.


## 4 · The Pigeonhole Principle

**Theorem (Pigeonhole Principle).** If $|A| > |B|$, then no injective function
from $A$ to $B$ exists. Equivalently: if $n$ items are placed in $k < n$ containers,
at least one container holds more than one item.

**Formal statement:**

$$|A| > |B| \;\Rightarrow\; \nexists\, f: A \to B \text{ injective}$$

In other words: if you try to map a larger set into a smaller one, you must have
collisions — two elements of $A$ that map to the same element of $B$.

### Cryptographic Applications

**Hash collisions are guaranteed.** A cryptographic hash function
$H: \{0,1\}^* \to \{0,1\}^{256}$ maps arbitrarily long inputs to 256-bit outputs.
The input set is infinite; the output set has $2^{256}$ elements.

By the Pigeonhole Principle: infinitely many inputs share every output.
Collisions (two inputs with the same hash) are mathematically unavoidable.

The security question is not *whether* collisions exist — they always do — but
*how hard they are to find*. A collision-resistant hash function makes finding
any collision computationally infeasible, even though infinitely many exist.

**Birthday Bound.** In a set of $n = 2^{256}$ outputs, you expect to find a
collision after approximately $\sqrt{n} = 2^{128}$ random hash evaluations.
This is the **birthday paradox** — the Pigeonhole Principle applied probabilistically.
SHA-256 is designed so that $2^{128}$ operations is still infeasible.

**Session token collisions.** If a server issues 32-bit session tokens, the
token space has $2^{32} \approx 4 \times 10^9$ values. After issuing
$\sim 2^{16} = 65{,}536$ tokens, you expect a collision with ~50% probability
(birthday bound). A 32-bit token space is dangerously small for a large service.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[CS\]

| Concept | Cryptographic meaning | Broken when... |
|---|---|---|
| **Bijection** | Encryption function $E_\kappa$ | If not injective: two plaintexts → same ciphertext; decryption ambiguous |
| **Inverse** | Decryption function $D_\kappa = E_\kappa^{-1}$ | If $E_\kappa$ not surjective: some ciphertexts have no plaintext |
| **Composition** | Multi-round cipher = chain of bijections | Weak composition → structural attacks |
| **Permutation** | S-box (substitution) or P-layer (bit rearrangement) | Low non-linearity → linear cryptanalysis |
| **$|\mathcal{K}| = 2^k$** | Key space cardinality | $k < 80$ bits → brute-force feasible |
| **Pigeonhole** | Hash collisions are unavoidable | Short output → birthday attack feasible |
| **$\sqrt{|\text{output set}|}$** | Birthday bound for collision search | Output $< 128$ bits → collision search feasible |

**The core theorem of symmetric cryptography,** stated in the language of
this module:

> *A secure block cipher is a pseudorandom permutation family
> $\{E_\kappa\}_{\kappa \in \mathcal{K}}$ where each $E_\kappa: \mathcal{M} \to \mathcal{C}$
> is a bijection, and no efficient algorithm can distinguish
> $E_\kappa$ from a uniformly random permutation given oracle access.*

Every term in that statement — family, bijection, uniform distribution over the
symmetric group — is now precise for you.

---


## Python: Visualization & Verification

**1 · Bijection Prover** — implement formal checks for injectivity and surjectivity;
prove the byte complement and a toy S-box are bijections; visualise the mapping.

**2 · Toy Substitution-Permutation Network** — build a miniature SPN operating
on 8-bit blocks using a toy S-box and a bit-permutation layer; visualise one
round of encryption and verify the full cipher is a bijection.

**3 · Pigeonhole and the Birthday Bound** — demonstrate hash collision inevitability
on a small output space; plot the birthday collision probability curve; compute
the expected number of evaluations for a collision at various output sizes.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.colors import ListedColormap
import hashlib, random

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Bijection Prover and Mapping Visualizer

We implement formal injectivity and surjectivity checks, then apply them to
three candidate functions on the byte set $\mathbb{B}$:

1. **Byte complement** $f(x) = 255 - x$ — bijection (proved in Section 1)
2. **Toy AES-style S-box** — a lookup table; bijection iff no repeated values
3. **Broken S-box** — a deliberately non-injective mapping (collision exists)

We then visualise how the bijective S-box permutes the 256 byte values.


In [ ]:
# ── 1 · Bijection Prover and S-box Visualizer ─────────────────────────────────
random.seed(0)

BYTE = list(range(256))

def is_injective(f_dict):
    """f_dict: {input: output}. Injective iff all outputs distinct."""
    outputs = list(f_dict.values())
    return len(outputs) == len(set(outputs))

def is_surjective(f_dict, codomain):
    """Surjective iff every codomain element appears as an output."""
    return set(f_dict.values()) == set(codomain)

def is_bijective(f_dict, codomain):
    return is_injective(f_dict) and is_surjective(f_dict, codomain)

# Byte complement
f_complement = {x: 255 - x for x in BYTE}

# Toy S-box: a random permutation of 0..255
toy_sbox_list = list(range(256))
random.shuffle(toy_sbox_list)
f_sbox = {x: toy_sbox_list[x] for x in BYTE}

# Broken S-box: non-injective (output 0 maps to many inputs)
f_broken = dict(f_sbox)
f_broken[0]   = 42   # collision: both 0 and original key for 42 map to 42
f_broken[128] = 42

funcs = [
    ("Byte complement  f(x)=255−x",  f_complement),
    ("Toy S-box  (random permutation)", f_sbox),
    ("Broken S-box  (collision at 42)",  f_broken),
]

print(f"{'Function':<42} {'Injective':^10} {'Surjective':^11} {'Bijective':^10}")
print("─" * 76)
for name, f in funcs:
    inj = is_injective(f)
    sur = is_surjective(f, BYTE)
    bij = inj and sur
    marks = [('✓' if v else '✗') for v in [inj, sur, bij]]
    print(f"  {name:<40} {'  '.join(f'{m:^10}' for m in marks)}")

# ── Visualise the S-box mapping ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: scatter plot of f_sbox (input → output)
ax = axes[0]
xs = list(f_sbox.keys())
ys = list(f_sbox.values())
ax.scatter(xs, ys, s=4, color=TS_BLUE, alpha=0.7, zorder=3)
ax.plot([0, 255], [0, 255], color=TS_GRAY, lw=1, linestyle='--',
        alpha=0.4, label='y = x  (identity)')
ax.set_xlabel('Input byte $x$')
ax.set_ylabel('Output byte $\sigma(x)$')
ax.set_title('Toy S-box: Input → Output
(bijection — every output reached exactly once)',
             fontsize=9, fontweight='bold')
ax.legend(fontsize=8)

# Right: output histogram — should be perfectly flat for a bijection
ax2 = axes[1]
output_counts = np.zeros(256, dtype=int)
for v in f_sbox.values():
    output_counts[v] += 1
ax2.bar(range(256), output_counts, color=TS_GREEN, width=1.0, zorder=3)

broken_counts = np.zeros(256, dtype=int)
for v in f_broken.values():
    broken_counts[v] += 1
collision_mask = broken_counts > 1
ax2.bar(np.where(collision_mask)[0], broken_counts[collision_mask],
        color=TS_RED, width=1.0, zorder=4, label='Collision (broken S-box)')

ax2.axhline(1, color=TS_GRAY, lw=1.5, linestyle='--', label='Expected: 1 (bijection)')
ax2.set_xlabel('Output byte value')
ax2.set_ylabel('Number of inputs mapping to this output')
ax2.set_title('Output Frequency Histogram
'
              'Flat = bijection  |  Spike = collision (non-injective)',
              fontsize=9, fontweight='bold')
ax2.legend(fontsize=8)
ax2.set_ylim(0, 4)

plt.suptitle('Bijection Analysis of S-box Functions', fontsize=12,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Formal collision report
collisions = {k: v for k, v in f_broken.items()
              if list(f_broken.values()).count(v) > 1}
print(f"\nBroken S-box collision: output value 42 reached by inputs:")
print(f"  {[k for k,v in f_broken.items() if v == 42]}")
print(f"  → NOT injective → NOT a valid encryption function")


**What do you see?**

- The scatter plot of the toy S-box has no two points in the same column or row —
  each input maps to a unique output (injective) and every output is reached (surjective).
- The histogram is perfectly flat at height 1 for the valid S-box — every byte
  value appears exactly once as an output. This is the visual signature of a bijection.
- The red spike at output 42 in the histogram marks the broken S-box collision —
  two inputs map to 42, and output 42 from the original appears nowhere.
- The classification table confirms: broken S-box is neither injective nor surjective.

**Practical implication.** If an encryption function has a collision — two plaintexts
mapping to the same ciphertext — decryption is ambiguous: given the ciphertext, you
cannot determine which plaintext was encrypted. The bijection requirement is not a
mathematical nicety; it is the condition that makes decryption possible at all.


### 2 · Toy Substitution-Permutation Network (SPN)

We build a miniature SPN operating on 8-bit blocks — small enough to visualise
completely. One round consists of:

1. **Key mixing** — XOR the block with a round key ($\oplus$, *(exclusive or)*)
2. **Substitution** — apply a 4-bit S-box to each nibble (the two 4-bit halves)
3. **Permutation** — rearrange the 8 bit positions by a fixed permutation

We run a plaintext through one round of encryption, track every intermediate value,
and verify the full cipher (over all 256 possible inputs) is a bijection.


In [ ]:
# ── 2 · Toy Substitution-Permutation Network ─────────────────────────────────

# 4-bit S-box (16 entries, a bijection on {0,...,15})
SBOX_4 = [0xC, 0x5, 0x6, 0xB, 0x9, 0x0, 0xA, 0xD,
           0x3, 0xE, 0xF, 0x8, 0x4, 0x7, 0x1, 0x2]

# Inverse S-box (so we can decrypt)
SBOX_4_INV = [0] * 16
for i, v in enumerate(SBOX_4):
    SBOX_4_INV[v] = i

# Bit permutation for 8-bit block: position i → position P[i]
P_LAYER = [1, 5, 2, 0, 3, 7, 4, 6]  # maps bit position i to P_LAYER[i]

def apply_sbox(byte_val):
    """Apply 4-bit S-box to both nibbles of a byte."""
    hi = (byte_val >> 4) & 0xF
    lo = byte_val & 0xF
    return (SBOX_4[hi] << 4) | SBOX_4[lo]

def apply_perm(byte_val):
    """Apply bit permutation to 8-bit value."""
    bits_in = [(byte_val >> (7 - i)) & 1 for i in range(8)]
    bits_out = [0] * 8
    for i, bit in enumerate(bits_in):
        bits_out[P_LAYER[i]] = bit
    return int(''.join(str(b) for b in bits_out), 2)

def spn_round(plaintext, round_key):
    """One SPN round: key mixing → substitution → permutation."""
    after_key  = plaintext ^ round_key
    after_sub  = apply_sbox(after_key)
    after_perm = apply_perm(after_sub)
    return after_key, after_sub, after_perm

# ── Trace one encryption ───────────────────────────────────────────────────────
plaintext  = 0b10110100   # 0xB4
round_key  = 0b01101001   # 0x69

after_key, after_sub, after_perm = spn_round(plaintext, round_key)

print("SPN Round Trace")
print("=" * 50)
print(f"  Plaintext      : {plaintext:08b}  (0x{plaintext:02X})")
print(f"  Round key      : {round_key:08b}  (0x{round_key:02X})")
print(f"  After XOR      : {after_key:08b}  (0x{after_key:02X})")
print(f"  After S-box    : {after_sub:08b}  (0x{after_sub:02X})")
print(f"  After P-layer  : {after_perm:08b}  (0x{after_perm:02X})  ← ciphertext")

# Verify full cipher is a bijection over all 256 inputs
full_cipher = {pt: spn_round(pt, round_key)[2] for pt in range(256)}
print(f"\nFull cipher over all 256 inputs:")
print(f"  Injective:   {is_injective(full_cipher)}  (no two plaintexts → same ciphertext)")
print(f"  Surjective:  {is_surjective(full_cipher, list(range(256)))}  (every ciphertext reachable)")
print(f"  Bijective:   {is_bijective(full_cipher, list(range(256)))}  ✓  decryption always possible")

# ── Visualise the SPN round as a diagram ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(-0.5, 10)
ax.set_ylim(-0.5, 4)
ax.axis('off')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')

def draw_block(ax, x, y, label, value_bin, value_hex, fc, ec):
    rect = mpatches.FancyBboxPatch(
        (x - 1.1, y - 0.38), 2.2, 0.76,
        boxstyle='round,pad=0.08', facecolor=fc, edgecolor=ec, linewidth=2
    )
    ax.add_patch(rect)
    ax.text(x, y + 0.12, f'{value_bin}', ha='center', va='center',
            fontsize=9, fontweight='bold', color='white', fontfamily='monospace')
    ax.text(x, y - 0.15, f'0x{value_hex:02X}  ({label})', ha='center', va='center',
            fontsize=7.5, color='white')

def draw_op(ax, x, y, symbol, fc):
    circle = plt.Circle((x, y), 0.28, color=fc, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, symbol, ha='center', va='center', fontsize=13,
            fontweight='bold', color='white', zorder=4)

def draw_arrow(ax, x0, y0, x1, y1):
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle='->', color=TS_GRAY, lw=1.8))

y = 1.8
draw_block(ax, 1.3, y, 'plaintext', f'{plaintext:08b}', plaintext, TS_BLUE, TS_BLUE)
draw_arrow(ax, 2.4, y, 3.1, y)
draw_op(ax, 3.4, y, '⊕', TS_AMBER)
draw_arrow(ax, 3.7, y, 4.3, y)
draw_block(ax, 5.4, y, 'after XOR', f'{after_key:08b}', after_key, TS_AMBER, TS_AMBER)
draw_arrow(ax, 6.5, y, 7.1, y)
draw_op(ax, 7.4, y, 'S', TS_GREEN)
draw_arrow(ax, 7.7, y, 8.1, y)
draw_block(ax, 9.2, y, 'after S', f'{after_sub:08b}', after_sub, TS_GREEN, TS_GREEN)

y2 = 0.5
draw_block(ax, 9.2, y2, 'after P', f'{after_perm:08b}', after_perm, TS_RED, TS_RED)
draw_arrow(ax, 9.2, 1.42, 9.2, 0.88)
draw_op(ax, 9.2, 1.15, 'P', TS_RED)

# Key below XOR
ax.text(3.4, 0.8, f'round key
{round_key:08b} (0x{round_key:02X})',
        ha='center', fontsize=8, color=TS_AMBER, style='italic')
draw_arrow(ax, 3.4, 1.08, 3.4, 1.52)

ax.set_title('One Round of the Toy SPN
'
             'Key mixing (⊕) → Substitution (S-box) → Permutation (P-layer)',
             fontsize=11, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()


**What do you see?**

The diagram shows one SPN round operating on an 8-bit block:

1. **XOR with round key** ($\oplus$, amber): mixes the key material into the block —
   this is the only step that depends on the secret key.
2. **S-box substitution** (S, green): applies the non-linear 4-bit lookup to each
   nibble — this is what makes the cipher resistant to linear cryptanalysis.
3. **Bit permutation** (P, red): rearranges bit positions — this is the *diffusion*
   step that spreads each bit's influence across the whole block.

The verification at the bottom confirms: the full round cipher, applied to all 256
possible inputs with a fixed key, is a bijection. Every plaintext maps to a unique
ciphertext, and every ciphertext corresponds to exactly one plaintext.

**The AES connection.** AES-128 is a 10-round SPN operating on 128-bit blocks.
Its S-box is a bijection on $\{0,\ldots,255\}$ constructed from the multiplicative
inverse in $GF(2^8)$ (a field we will study in Module 08). Its diffusion layer
(MixColumns + ShiftRows) spreads bit influence across the full 128-bit state.
Every round is bijective; the composition of 10 bijective rounds is bijective.


### 3 · The Pigeonhole Principle and the Birthday Bound

We demonstrate the Pigeonhole Principle concretely on a small output space,
then derive and plot the birthday collision probability curve to show why short
hash outputs are insecure.


In [ ]:
# ── 3 · Pigeonhole Principle and Birthday Bound ───────────────────────────────

# ── Pigeonhole on a tiny hash space ────────────────────────────────────────────
OUTPUT_BITS = 8   # only 256 possible hash values
OUTPUT_SIZE = 2 ** OUTPUT_BITS

random.seed(42)

def tiny_hash(value, bits=OUTPUT_BITS):
    """A truncated hash to 'bits' bits — tiny output space."""
    h = hashlib.sha256(str(value).encode()).digest()
    full = int.from_bytes(h, 'big')
    return full % (2 ** bits)

# Search for a collision
seen = {}
collision = None
for attempt in range(1, 10000):
    val    = random.randint(0, 10**9)
    digest = tiny_hash(val)
    if digest in seen:
        collision = (seen[digest], val, digest)
        break
    seen[digest] = val

print(f"Tiny hash: output space = {OUTPUT_SIZE} values (2^{OUTPUT_BITS} bits)")
print(f"Collision found after {attempt} evaluations")
print(f"  Input A = {collision[0]}  →  hash = {collision[2]}")
print(f"  Input B = {collision[1]}  →  hash = {collision[2]}")
print(f"  Both map to the same {OUTPUT_BITS}-bit output — Pigeonhole guarantees this exists!")
print(f"  Birthday bound estimate: ≈ sqrt({OUTPUT_SIZE}) = {int(OUTPUT_SIZE**0.5)}")

# ── Birthday probability curve ────────────────────────────────────────────────
def birthday_prob(n, k):
    """Probability of at least one collision after k draws from n values."""
    # Approximation: 1 - e^(-k(k-1)/(2n))
    return 1 - np.exp(-k * (k - 1) / (2 * n))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: probability curve for our tiny hash
ax = axes[0]
ks = np.arange(1, 3 * int(OUTPUT_SIZE**0.5) + 1)
probs = birthday_prob(OUTPUT_SIZE, ks)

ax.plot(ks, probs, color=TS_BLUE, lw=2.5)
ax.axhline(0.5, color=TS_AMBER, lw=1.5, linestyle='--',
           label=f'50% collision at k ≈ {int(OUTPUT_SIZE**0.5)}')
ax.axvline(int(OUTPUT_SIZE**0.5), color=TS_AMBER, lw=1.5, linestyle='--')
ax.scatter([attempt], [birthday_prob(OUTPUT_SIZE, attempt)],
           color=TS_RED, s=120, zorder=5, label=f'Actual collision at k={attempt}')
ax.set_xlabel('Number of hash evaluations $k$')
ax.set_ylabel('Probability of at least one collision')
ax.set_title(f'Birthday Curve — {OUTPUT_BITS}-bit output space ({OUTPUT_SIZE} values)',
             fontsize=9, fontweight='bold')
ax.legend(fontsize=8)
ax.set_ylim(0, 1.05)

# Right: birthday bound vs output size
ax2 = axes[1]
bit_sizes = np.arange(8, 260, 4)
birthday_bounds = 2 ** (bit_sizes / 2)

ax2.semilogy(bit_sizes, birthday_bounds, color=TS_BLUE, lw=2.5)
ax2.fill_between(bit_sizes, 1, birthday_bounds, alpha=0.1, color=TS_BLUE)

milestones = [(32, '2^16 ≈ 65K
(broken)'),
              (64, '2^32 ≈ 4B'),
              (128, '2^64
(infeasible)'),
              (256, '2^128
(SHA-256)')]
for bits, label in milestones:
    y = 2 ** (bits / 2)
    ax2.scatter([bits], [y], color=TS_AMBER, s=100, zorder=5)
    ax2.annotate(label, xy=(bits, y), xytext=(bits + 8, y * 5),
                 fontsize=8, color=TS_AMBER,
                 arrowprops=dict(arrowstyle='->', color=TS_AMBER, lw=1))

ax2.set_xlabel('Hash output size (bits)')
ax2.set_ylabel('Birthday bound — evaluations for 50% collision')
ax2.set_title('Birthday Bound vs Hash Output Size
'
              'Security requires birthday bound >> computational budget',
              fontsize=9, fontweight='bold')
ax2.grid(True, which='both', alpha=0.3)

plt.suptitle('Pigeonhole Principle and the Birthday Attack',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**What do you see?**

- The collision is found in far fewer than 256 evaluations — the birthday bound
  predicts ~16 evaluations for 50% probability on a 256-value space. The actual
  collision confirms the Pigeonhole Principle is not merely theoretical.

- The birthday probability curve rises steeply from 0 to 1, crossing 50% right
  at $\sqrt{2^8} = 2^4 = 16$ evaluations (the orange dashed lines).

- The right plot shows why 32-bit hash outputs (birthday bound $2^{16} \approx 65{,}000$)
  are broken for collision resistance — that's a few seconds of computation. The
  128-bit birthday bound of $2^{64}$ operations remains infeasible; SHA-256's
  $2^{128}$ bound is far beyond any foreseeable computational capacity.

**The lesson from set theory.** The Pigeonhole Principle is a theorem about
*cardinalities of sets*: when the domain is larger than the codomain, the
function cannot be injective. The entire science of collision resistance is an
application of this single set-theoretic fact.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. INVERSE OF THE TOY S-BOX
#    The toy S-box SBOX_4 is a bijection on {0,...,15}.
#    Compute its inverse SBOX_4_INV (a list of 16 values).
#    Verify: for all x in range(16), SBOX_4_INV[SBOX_4[x]] == x
#    Then build the full inverse SPN round: reverse the P-layer,
#    apply inverse S-box, XOR the round key again.
#    Verify: inverse_round(spn_round(pt, key)[2], key) == pt for all pt.
#
# 2. MULTI-ROUND SPN
#    Extend the toy SPN to 3 rounds with three different round keys.
#    Verify the 3-round cipher is still a bijection over all 256 inputs.
#    Plot: for each pair of rounds, compare how much the output changes
#    when one bit of the input flips — this is the "avalanche effect."
#
# 3. KEY SPACE ANALYSIS
#    For the toy SPN with 8-bit keys and 8-bit blocks:
#      - How many distinct ciphers are possible? (= key space size)
#      - How does this compare to |S_256|, the total number of
#        bijections on {0,...,255}?
#      - Express both as powers of 2 and comment on the gap.
#    (Hint: |S_256| = 256! ≈ 2^1684)

# Your work here:


---

## Summary

| Concept | Definition | Cryptographic meaning |
|---|---|---|
| **Bijection** | Injective + surjective | Encryption must be invertible |
| **Inverse** | $f^{-1} \circ f = \text{id}$ | Decryption undoes encryption exactly |
| **Composition** | $(g \circ f)(x) = g(f(x))$ | Multi-round cipher = chain of bijections |
| **Permutation** | Bijection $A \to A$ | S-box and P-layer in block ciphers |
| **Key space $|\mathcal{K}| = 2^k$** | All possible keys | Security ∝ cost of exhaustive search |
| **Pigeonhole Principle** | $|A| > |B| \Rightarrow$ no injection exists | Hash collisions are mathematically unavoidable |
| **Birthday bound** | $\approx \sqrt{|\text{output set}|}$ | Evaluations to find a collision with 50% probability |

---

## Module 03 Complete

| Unit | Content | Payoff |
|---|---|---|
| **01** | Sets, operations, power sets | Formal language for permission sets, input domains, attack surfaces |
| **02** | Relations, properties, functions | Access control matrices; type systems; trust zone partitions |
| **03** | Bijections, permutations, crypto | Block cipher structure; key spaces; hash collision theory |

The three units form a single argument: the objects security systems operate on
are sets; the policies are relations over those sets; the cryptographic primitives
securing those policies are bijective functions on those sets.

---

## Up Next

**Module 04 — Graph Theory I: Structure**

Sets with relations are the foundation. When the relation is between vertices
connected by edges, the structure becomes a **graph** — the native language of
neural network architecture, computation graphs, and agent tool call networks.
Module 04 builds graph theory from first principles.

$\rightarrow$ `modules/module-04/unit-01-graphs-definitions.ipynb`
